# statistical analysis

**Purpose**: aggregate the results/runs/main_matrix/ matrix, compute mean ± std per (backbone, pair, method), run Wilcoxon signed-rank tests (each sophisticated method vs Random-K=2), bootstrap 95% CIs, and generate the main paper Sec. 4.1 / 5.3 / 5.4 tables.

**Runs on**:  local Docker or Colab CPU (no GPU needed).

**Prerequisite**: main matrix has finished running Phase 1-3 results and pushed to git.


## 1. Setup

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy import stats

# Auto-detect base path (Mac or Colab)
CANDIDATES = [
    Path('$REPO_ROOT/Documents/git/analysis'),
    Path('/content/drive/MyDrive/cd-er-paradigm-choice'),
    Path.cwd(),
]
for c in CANDIDATES:
    if (c / 'results/runs/main_matrix').exists():
        BASE = c
        break
else:
    BASE = CANDIDATES[0]
    print(f'[warn] no main matrix results yet, using {BASE}')

print(f'BASE: {BASE}')
RUN_DIR = BASE / 'results/runs/main_matrix'

## 2. Load all metrics.json into a DataFrame

In [ ]:
rows = []
if RUN_DIR.exists():
    for metrics_file in RUN_DIR.rglob('metrics.json'):
        try:
            d = json.load(open(metrics_file))
            if d.get('test_f1') is not None:
                rows.append({
                    'backbone': d.get('backbone'),
                    'source': d.get('source_dataset'),
                    'target': d.get('target_dataset'),
                    'method': d.get('method'),
                    'seed': d.get('seed'),
                    'f1': d.get('test_f1'),
                    'precision': d.get('test_precision'),
                    'recall': d.get('test_recall'),
                    'n_test': d.get('n_test_pairs'),
                    'pos_rate': d.get('positive_rate'),
                    'elapsed_sec': d.get('elapsed_sec'),
                })
        except Exception as e:
            print(f'[err] {metrics_file}: {e}')

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} cells')
if not df.empty:
    print(f'Backbones: {sorted(df.backbone.unique())}')
    print(f'Pairs: {sorted(df.target.unique())}')
    print(f'Methods: {sorted(df.method.unique())}')
    print(f'Seeds: {sorted(df.seed.unique())}')
df.head(10)

## 3. Mean ± std per (backbone, target, method)

In [ ]:
if df.empty:
    print('No results yet — run main matrix first.')
else:
    agg = df.groupby(['backbone', 'target', 'method'])['f1'].agg(['mean', 'std', 'count']).round(4)
    agg.columns = ['f1_mean', 'f1_std', 'n_seeds']
    print(agg.to_string())

## 4. LLM K-curve per backbone (main paper Sec. 4.1)

In [ ]:
if df.empty:
    print('empty')
else:
    # Wide table: rows = (backbone, target), columns = method
    pivot = df.pivot_table(index=['backbone', 'target'], columns='method',
                            values='f1', aggfunc=lambda x: f'{x.mean():.3f}±{x.std():.3f}')
    print('main paper Table 3 — mean ± std across 3 seeds')
    print(pivot.to_string())

## 5. Wilcoxon signed-rank tests: each sophisticated method vs Random-K=2

In [ ]:
if df.empty:
    print('empty')
else:
    # For each (backbone), compute Wilcoxon on per-(target, seed) F1 pairs:
    # sophisticated method vs random_k2
    print(f'{"backbone":<15} {"method":<12} {"n_pairs":>8} {"mean_delta":>10} {"p_value":>10}')
    print('-' * 60)
    for backbone in sorted(df.backbone.unique()):
        for method in sorted(df.method.unique()):
            if method == 'random_k2':
                continue
            d1 = df[(df.backbone == backbone) & (df.method == 'random_k2')].sort_values(['target', 'seed'])
            d2 = df[(df.backbone == backbone) & (df.method == method)].sort_values(['target', 'seed'])
            # Pair on (target, seed)
            m = d1.merge(d2, on=['target', 'seed'], suffixes=('_random', '_soph'))
            if len(m) < 3:
                continue
            deltas = m['f1_soph'] - m['f1_random']
            try:
                stat, pval = stats.wilcoxon(deltas)
            except ValueError:
                pval = float('nan')
            print(f'{backbone:<15} {method:<12} {len(m):>8} {deltas.mean():>+10.4f} {pval:>10.4f}')

## 6. Bootstrap 95% CI on aggregate Δ (each sophisticated method vs random)

In [ ]:
def bootstrap_ci(values, n_boot=10000, alpha=0.05, rng_seed=42):
    rng = np.random.default_rng(rng_seed)
    n = len(values)
    boot_means = np.empty(n_boot)
    values = np.asarray(values)
    for i in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        boot_means[i] = sample.mean()
    lo = np.quantile(boot_means, alpha / 2)
    hi = np.quantile(boot_means, 1 - alpha / 2)
    return float(values.mean()), float(lo), float(hi)


if not df.empty:
    print(f'{"backbone":<15} {"method":<12} {"n":>4} {"mean_delta":>10} {"95% CI":>25}')
    print('-' * 70)
    for backbone in sorted(df.backbone.unique()):
        for method in sorted(df.method.unique()):
            if method == 'random_k2':
                continue
            d1 = df[(df.backbone == backbone) & (df.method == 'random_k2')].sort_values(['target', 'seed'])
            d2 = df[(df.backbone == backbone) & (df.method == method)].sort_values(['target', 'seed'])
            m = d1.merge(d2, on=['target', 'seed'], suffixes=('_random', '_soph'))
            if len(m) < 3:
                continue
            deltas = (m['f1_soph'] - m['f1_random']).values
            mean, lo, hi = bootstrap_ci(deltas)
            print(f'{backbone:<15} {method:<12} {len(m):>4} {mean:>+10.4f}   [{lo:+.4f}, {hi:+.4f}]')

## 7. Effect-size (Cohen's d) alongside p-value

In [ ]:
def cohens_d_paired(x1, x2):
    diffs = np.asarray(x1) - np.asarray(x2)
    if len(diffs) < 2 or diffs.std() == 0:
        return 0.0
    return diffs.mean() / diffs.std()


if not df.empty:
    print(f'{"backbone":<15} {"method":<12} {"n":>4} {"cohen_d":>10}')
    print('-' * 45)
    for backbone in sorted(df.backbone.unique()):
        for method in sorted(df.method.unique()):
            if method == 'random_k2':
                continue
            d1 = df[(df.backbone == backbone) & (df.method == 'random_k2')].sort_values(['target', 'seed'])
            d2 = df[(df.backbone == backbone) & (df.method == method)].sort_values(['target', 'seed'])
            m = d1.merge(d2, on=['target', 'seed'], suffixes=('_random', '_soph'))
            if len(m) < 3:
                continue
            d = cohens_d_paired(m['f1_soph'], m['f1_random'])
            print(f'{backbone:<15} {method:<12} {len(m):>4} {d:>+10.3f}')

## 8. K-sweep robustness (needs separate main matrix run with K∈{0,1,2,4,8,10})

In [ ]:
# Assumes results/runs/main_matrix-ksweep/ exists after running K-sweep
KSWEEP_DIR = BASE / 'results/runs/main_matrix-ksweep'
if not KSWEEP_DIR.exists():
    print('K-sweep results not yet available. Run K-sweep cell in main matrix first.')
else:
    ksweep_rows = []
    for m in KSWEEP_DIR.rglob('metrics.json'):
        try:
            d = json.load(open(m))
            ksweep_rows.append({'k': d.get('k_demos', 0), 'seed': d.get('seed'),
                                 'f1': d.get('test_f1')})
        except: pass
    kdf = pd.DataFrame(ksweep_rows)
    print(kdf.groupby('k')['f1'].agg(['mean', 'std', 'count']).round(4))

## 9. Save aggregate table for main paper Sec. 5

In [ ]:
if not df.empty:
    out_dir = BASE / 'docs/paper/artifacts'
    out_dir.mkdir(exist_ok=True)
    df.to_csv(out_dir / 'main_matrix_raw.csv', index=False)
    agg = df.groupby(['backbone', 'target', 'method'])['f1'].agg(['mean', 'std', 'count']).round(4)
    agg.to_csv(out_dir / 'main_matrix_aggregated.csv')
    print(f'[saved] {out_dir}/main_matrix_raw.csv')
    print(f'[saved] {out_dir}/main_matrix_aggregated.csv')

## 10. Generate main paper Sec. 4.1 Table 3

In [ ]:
if not df.empty:
    # main paper Table 3: rows=backbone×target, columns=method (F1 mean±std)
    methods_order = ['random_k2', 'kate_k2', 'jac_k2', 'cider_k2']
    method_display = {'random_k2': 'Random-strat (K=2)', 'kate_k2': 'KATE (K=2)',
                       'jac_k2': 'JAC (K=2)', 'cider_k2': 'CIDER-strat (K=2)'}
    backbone_display = {'llama-3.3-70b': 'LLaMA-3.3-70B', 'llama-3.1-8b': 'Llama-3.1-8B',
                         'qwen-2.5-72b': 'Qwen2.5-72B', 'gpt-4o-mini': 'GPT-4o-mini'}

    lines = ['| Backbone | Target | ' + ' | '.join(method_display.get(m, m) for m in methods_order) + ' |',
             '|' + '---|' * (len(methods_order) + 2)]
    for backbone in sorted(df.backbone.unique()):
        for target in sorted(df.target.unique()):
            cells_ = []
            for m in methods_order:
                sub = df[(df.backbone == backbone) & (df.target == target) & (df.method == m)]
                if len(sub) > 0:
                    mean, std = sub.f1.mean(), sub.f1.std()
                    cells_.append(f'{mean:.3f}±{std:.3f}')
                else:
                    cells_.append('—')
            lines.append(f'| {backbone_display.get(backbone, backbone)} | {target} | ' +
                          ' | '.join(cells_) + ' |')
    print('\n'.join(lines))